# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação
### 🎯 Laboratório Prático: Detecção de Objetos Multiclasse com Faster R-CNN (Dataset Kaggle via `kagglehub`, Fine-Tuning e Desmistificação de Saídas)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_01_cnn_architectures/aula_01_faster_rcnn.ipynb)

---

### 📖 Contextualização Pedagógica

Na classificação de imagens tradicional, a rede convolucional responde à pergunta: **"O que há nesta imagem?"**, gerando um único rótulo para todo o quadro visual. 

No entanto, na detecção de objetos do mundo real (varejo inteligente, colheita robótica, veículos autônomos e diagnóstico médico), precisamos responder: **"Quais objetos estão na imagem, de quais classes e exatamente ONDE estão localizados?"**.

Neste laboratório prático, utilizaremos a biblioteca oficial **`kagglehub`** para baixar instantaneamente um dataset multiclasse de imagens reais do Kaggle (**Fruit Images for Object Detection**) e faremos o *fine-tuning* da arquitetura canônica **Faster R-CNN ResNet-50 FPN V2 (Two-Stage)**.

---

### 🎯 Objetivos de Aprendizagem
1. **Download Ultrarrápido via `kagglehub`**: Integrar a API moderna do Kaggle sem downloads manuais ou configurações complexas.
2. **Construir um Dataset de Detecção em Pascal VOC XML**: Implementar a classe `Dataset` com anotações de *Bounding Boxes* `[x1, y1, x2, y2]` e um `collate_fn` para imagens com múltiplos objetos.
3. **Compreender a Anatomia do Faster R-CNN**: RPN (*Region Proposal Network*), *RoIAlign* e a Cabeça *Fast R-CNN Densa*.
4. **Carregar e Adaptar um Modelo Pré-Treinado Moderno**: Utilizar a API de *Weights Enum* (`FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT`) e substituir cirurgicamente o `FastRCNNPredictor` para as classes de interesse.
5. **Desmistificar as Saídas do Modelo**: Inspecionar minuciosamente os tensores `boxes`, `labels` e `scores` no modo `eval()`, aplicando filtragem por *Score Threshold* e **Não-Supressão Máxima (NMS)**.
6. **Construir Visualizações Profissionais**: Renderizar caixas delimitadoras coloridas, rótulos e pontuações de confiança diretamente sobre imagens de teste.


## 1. Configuração do Ambiente e Instalação de Dependências

Vamos instalar a biblioteca `kagglehub` e verificar a GPU CUDA no Google Colab.


In [ ]:
# Instalar kagglehub para download ultrarrápido de datasets públicos do Kaggle
!pip install -q kagglehub

import os
import sys
import time
import math
import random
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as F
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2, 
    FasterRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Fixar sementes para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Dispositivo de execução
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Dispositivo de Execução: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   ⚠️ Atenção: Nenhuma GPU detectada. No Colab, vá em: Ambiente de Execução -> Alterar tipo de ambiente -> T4 GPU")


## 2. Download Automático do Dataset no Kaggle via `kagglehub`

Utilizaremos a função `kagglehub.dataset_download()` para baixar o dataset público **Fruit Images for Object Detection** (`mbkinaci/fruit-images-for-object-detection`).

### 📦 Vantagens do `kagglehub`:
- **Alta Velocidade:** O download é feito via CDN direta do Kaggle em poucos segundos.
- **Zero Configuração Manual:** Não exige upload de tokens ou chaves `kaggle.json` para datasets públicos abertos.
- **Cache Inteligente:** Se o dataset já tiver sido baixado na sessão, ele reutiliza o cache local instantaneamente.


In [ ]:
import kagglehub

print("📥 Baixando dataset do Kaggle via kagglehub...")
start_dl = time.time()

# Download do dataset 'mbkinaci/fruit-images-for-object-detection'
dataset_root = kagglehub.dataset_download("mbkinaci/fruit-images-for-object-detection")

print(f"✅ Download concluído em {time.time() - start_dl:.2f}s!")
print(f"📂 Diretório Local do Dataset: {dataset_root}")

# Localizar diretórios de treino e teste
# O dataset descompacta em pastas com sufixo _zip/train e _zip/test
train_dir = os.path.join(dataset_root, "train_zip", "train")
test_dir = os.path.join(dataset_root, "test_zip", "test")

# Caso a estrutura seja direta sem _zip:
if not os.path.exists(train_dir):
    train_dir = os.path.join(dataset_root, "train")
    test_dir = os.path.join(dataset_root, "test")

train_imgs = [f for f in os.listdir(train_dir) if f.endswith(('.jpg', '.png'))]
test_imgs = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png'))]

print(f"🍎 Imagens de Treinamento: {len(train_imgs)}")
print(f"🍌 Imagens de Teste:       {len(test_imgs)}")


## 3. Implementação do `Dataset` para Anotações Pascal VOC XML

O dataset utiliza o formato padrão de anotação **Pascal VOC XML**, onde cada imagem `.jpg` possui um arquivo `.xml` correspondente com tags `<object>` e `<bndbox>` contendo $[x_{\text{min}}, y_{\text{min}}, x_{\text{max}}, y_{\text{max}}]$.

### 🏷️ Mapeamento de Classes:
- `0`: **Background (Fundo)** — Obrigatório no TorchVision Detection
- `1`: **Maçã (Apple)**
- `2`: **Banana**
- `3`: **Laranja (Orange)**


In [ ]:
# Dicionário de Mapeamento de Classes
CLASSES_MAP = {
    'background': 0,
    'apple': 1,
    'banana': 2,
    'orange': 3
}

# Rótulos em Português e Cores para Plotagem
CLASSES_PT = {
    0: 'Fundo',
    1: 'Maçã',
    2: 'Banana',
    3: 'Laranja'
}

CLASS_COLORS = {
    1: '#EF4444', # Vermelho para Maçã
    2: '#EAB308', # Amarelo para Banana
    3: '#F97316'  # Laranja para Laranja
}

class KaggleFruitDetectionDataset(Dataset):
    def __init__(self, folder_path, classes_map=CLASSES_MAP, transforms=None):
        self.folder_path = folder_path
        self.classes_map = classes_map
        self.transforms = transforms
        
        # Listar todas as imagens que possuem arquivo XML correspondente
        all_files = os.listdir(folder_path)
        self.images = sorted([f for f in all_files if f.endswith(('.jpg', '.png', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.folder_path, img_name)
        xml_name = os.path.splitext(img_name)[0] + ".xml"
        xml_path = os.path.join(self.folder_path, xml_name)

        # 1. Carregar imagem RGB
        img = Image.open(img_path).convert("RGB")
        width_img, height_img = img.size

        boxes = []
        labels = []

        # 2. Fazer parse do XML Pascal VOC
        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()

            for obj in root.findall("object"):
                class_name = obj.find("name").text.lower().strip()
                if class_name in self.classes_map:
                    label = self.classes_map[class_name]
                    bndbox = obj.find("bndbox")
                    xmin = float(bndbox.find("xmin").text)
                    ymin = float(bndbox.find("ymin").text)
                    xmax = float(bndbox.find("xmax").text)
                    ymax = float(bndbox.find("ymax").text)

                    # Validações de coordenadas (evitar caixas inválidas fora da imagem)
                    xmin = max(0.0, xmin)
                    ymin = max(0.0, ymin)
                    xmax = min(float(width_img), xmax)
                    ymax = min(float(height_img), ymax)

                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(label)

        # Converter para Tensores PyTorch
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }

        # Converter imagem PIL para Tensor [3, H, W] no intervalo [0.0, 1.0]
        img_tensor = F.to_tensor(img)

        return img_tensor, target

# Função de Collate Customizada Obrigatória para Object Detection
def collate_fn(batch):
    return tuple(zip(*batch))

# Instanciar Datasets
train_dataset = KaggleFruitDetectionDataset(train_dir)
test_dataset = KaggleFruitDetectionDataset(test_dir)

print(f"✅ Datasets criados com sucesso!")
print(f"   • Treino: {len(train_dataset)} amostras")
print(f"   • Teste:  {len(test_dataset)} amostras")


## 4. Visualização e Inspeção dos Dados de Treino

Vamos plotar amostras reais do dataset mostrando como as **Bounding Boxes de cada fruta** e seus rótulos estão anotados.


In [ ]:
def plot_sample_with_boxes(image_tensor, target, title="Amostra do Dataset"):
    """
    Utilitário para plotar um tensor de imagem com suas bounding boxes ground truth.
    """
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    
    fig, ax = plt.subplots(1, figsize=(8, 6))
    ax.imshow(img_np)
    
    boxes = target["boxes"].cpu().numpy()
    labels = target["labels"].cpu().numpy()
    
    for box, label in zip(boxes, labels):
        xmin, ymin, xmax, ymax = box
        width = xmax - xmin
        height = ymax - ymin
        
        color = CLASS_COLORS.get(label, '#22C55E')
        class_name = CLASSES_PT.get(label, f"Classe {label}")
        
        # Desenhar caixa
        rect = patches.Rectangle(
            (xmin, ymin), width, height, 
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Tag com nome da fruta
        ax.text(
            xmin, ymin - 6, 
            f"{class_name} (Real)", 
            fontsize=9.5, fontweight='bold', color='#FFFFFF',
            bbox=dict(boxstyle='round,pad=0.25', facecolor=color, alpha=0.95, edgecolor='none')
        )
        
    ax.set_title(f"{title} • {len(boxes)} objeto(s) anotado(s)", fontsize=12, fontweight='bold', color='#0A345D')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Visualizar 2 exemplos do dataset de treino
sample_img1, sample_target1 = train_dataset[10]
plot_sample_with_boxes(sample_img1, sample_target1, "Amostra #10 de Treino")

sample_img2, sample_target2 = train_dataset[25]
plot_sample_with_boxes(sample_img2, sample_target2, "Amostra #25 de Treino")


## 5. Configuração dos `DataLoaders`

Para detecção de objetos, o `collate_fn` agrupa as imagens em uma tupla e os alvos em outra tupla, preservando os tensores de tamanho variável de cada imagem.


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2 if torch.cuda.is_available() else 0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2 if torch.cuda.is_available() else 0,
    collate_fn=collate_fn
)

print(f"📊 Batches de Treino por época: {len(train_loader)} (com batch_size=4)")
print(f"📊 Amostras de Teste:          {len(test_loader)}")


## 6. A Arquitetura Faster R-CNN: Anatomia e Adaptação Cirúrgica

A arquitetura **Faster R-CNN (Two-Stage)** opera através do seguinte fluxo:

```
[Imagem de Entrada [3, H, W]]
             │
             ▼
[Backbone ResNet-50 + FPN (Feature Pyramid Network)] ──> Mapas em Múltiplas Escalas (P2 a P5)
             │
             ├──> [1º Estágio: RPN (Region Proposal Network)]
             │           │
             │           ▼ Âncoras Multiescala -> Gera ~1000 a 2000 Propostas de Regiões
             │           │
             ▼           ▼
       [RoI Align (7x7 Pooling Bilinear)] ──> Extrai Tensores de Features Alinhados
                         │
                         ▼
       [2º Estágio: Cabeça Densa Fast R-CNN (1024-D)]
             ├──> Classificador Softmax (C = 4: Fundo, Maçã, Banana, Laranja)
             └──> Regressor de Caixas [x1, y1, x2, y2] com Smooth L1 Loss
```

---

### 🔧 Substituição Cirúrgica da Cabeça de Predição (`FastRCNNPredictor`):
O modelo pré-treinado no COCO possui 91 classes. Adaptamos a cabeça final para as nossas **4 classes**:


In [ ]:
def get_faster_rcnn_multiclass(num_classes=4):
    """
    Carrega Faster R-CNN ResNet-50 FPN V2 pré-treinado no MS-COCO
    e adapta a cabeça para num_classes (4 classes).
    """
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights)
    
    # Dimensão de entrada da cabeça densa (1024 dimensões)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Substituir por novo preditor com 4 classes
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

# Instanciar modelo e mover para GPU
model = get_faster_rcnn_multiclass(num_classes=4)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🤖 Modelo: Faster R-CNN ResNet-50 FPN V2")
print(f"   • Parâmetros Totais:    {total_params / 1e6:.2f} M")
print(f"   • Parâmetros Treináveis: {trainable_params / 1e6:.2f} M")
print(f"   • Número de Classes:    4 (0: Fundo, 1: Maçã, 2: Banana, 3: Laranja)")


## 7. Desmistificando as Saídas do Faster R-CNN: Treino vs Inferência

O comportamento do método `forward()` do Faster R-CNN muda automaticamente com base no modo ativo:

---

### 1️⃣ Modo de Treino (`model.train()`):
- **Entrada:** `images` (lista de tensores) + `targets` (lista de dicionários).
- **Saída:** Um **dicionário de 4 perdas multitarefa simultâneas**:
  $$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{classifier}} + \mathcal{L}_{\text{box\_reg}} + \mathcal{L}_{\text{objectness}} + \mathcal{L}_{\text{rpn\_box\_reg}}$$
  - `loss_classifier`: Perda Cross-Entropy da classe final (Softmax).
  - `loss_box_reg`: Perda Smooth L1 de regressão das coordenadas finais $[x_1, y_1, x_2, y_2]$.
  - `loss_objectness`: Perda binária da RPN (*Objeto vs Fundo* nas âncoras).
  - `loss_rpn_box_reg`: Perda de ajuste das propostas de região geradas pela RPN.

---

### 2️⃣ Modo de Inferência (`model.eval()`):
- **Entrada:** Apenas `images` (lista de tensores).
- **Saída:** Uma **lista de dicionários de predições** (um para cada imagem do batch):
  - `boxes` (`FloatTensor[N, 4]`): Coordenadas das caixas em pixels absolutos $[x_1, y_1, x_2, y_2]$.
  - `labels` (`Int64Tensor[N]`): ID inteiro da classe prevista ($1 = \text{Maçã}, 2 = \text{Banana}, 3 = \text{Laranja}$).
  - `scores` (`FloatTensor[N]`): Confiança Softmax $[0.0, 1.0]$ ordenada decrescente.

Vamos demonstrar a inspeção real dos tensores a seguir:


In [ ]:
# Demonstrando o comportamento do modelo em modo EVAL
model.eval()

with torch.no_grad():
    test_sample_img, _ = test_dataset[0]
    test_sample_img_device = test_sample_img.unsqueeze(0).to(device)
    
    predictions = model(test_sample_img_device)

print("🔍 Inspecionando o Dicionário de Saída no Modo EVAL (model.eval()):")
pred = predictions[0]
for k, v in pred.items():
    print(f"   • Chave '{k}': Tensor de formato {v.shape} | tipo {v.dtype} | dispositivo {v.device}")

print(f"\n📦 Total de Caixas Candidatas detectadas: {len(pred['boxes'])}")
print(f"⭐ Top 3 Confianças Iniciais (antes do fine-tuning): {pred['scores'][:3].cpu().numpy()}")


## 8. Loop de Treinamento e Fine-Tuning

Configuramos o otimizador **SGD com Momentum e Weight Decay** e um agendador **StepLR** para ajustar as taxas de aprendizado durante as épocas.


In [ ]:
# Hiperparâmetros de Treinamento
NUM_EPOCHS = 8
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# Otimizador com parâmetros treináveis
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# Agendador de Taxa de Aprendizado
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.2)

# Históricos para análise gráfica
loss_history = []
loss_components = {
    'loss_classifier': [],
    'loss_box_reg': [],
    'loss_objectness': [],
    'loss_rpn_box_reg': []
}

print(f"🚀 Iniciando Fine-Tuning do Faster R-CNN por {NUM_EPOCHS} épocas...")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_components = {k: 0.0 for k in loss_components.keys()}
    
    pbar = tqdm(train_loader, desc=f"Época {epoch+1}/{NUM_EPOCHS} [Treino]")
    
    for images, targets in pbar:
        # Mover imagens e alvos para GPU
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass (calcula automaticamente as 4 losses)
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass & Otimização
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        # Acumular estatísticas
        epoch_loss += losses.item()
        for k in loss_components.keys():
            if k in loss_dict:
                epoch_components[k] += loss_dict[k].item()
                
        pbar.set_postfix({'Loss Total': f"{losses.item():.4f}", 'LR': f"{optimizer.param_groups[0]['lr']:.6f}"})
        
    # Atualizar agendador de taxa de aprendizado
    lr_scheduler.step()
    
    # Médias da época
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    for k in loss_components.keys():
        loss_components[k].append(epoch_components[k] / len(train_loader))
        
    print(f"✅ Época [{epoch+1:02d}/{NUM_EPOCHS:02d}] — Loss Média: {avg_loss:.4f} (Tempo acumulado: {time.time() - start_time:.1f}s)")

print(f"\n🎉 Fine-Tuning concluído com sucesso em {time.time() - start_time:.2f} segundos!")


## 9. Análise das Curvas de Aprendizado Multitarefa

Vamos plotar a evolução da **Perda Total Global** e das **4 Perdas Componentes** para validar a convergência do modelo.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Perda Total
ax1.plot(range(1, NUM_EPOCHS + 1), loss_history, 'o-', color='#0A345D', linewidth=2.5, markersize=7)
ax1.set_title('Evolução da Perda Total Multitarefa (Loss Global)', fontsize=12, fontweight='bold', color='#0A345D')
ax1.set_xlabel('Época', fontsize=11)
ax1.set_ylabel('Loss Total', fontsize=11)
ax1.set_xticks(range(1, NUM_EPOCHS + 1))
ax1.grid(True, alpha=0.3)

# Gráfico 2: Decomposição das 4 Perdas
colors = {'loss_classifier': '#16A34A', 'loss_box_reg': '#EA580C', 'loss_objectness': '#0284C7', 'loss_rpn_box_reg': '#9333EA'}
labels_pt = {
    'loss_classifier': 'Classificador Fast R-CNN',
    'loss_box_reg': 'Regressão BBox Fast R-CNN',
    'loss_objectness': 'RPN Objectness (Fundo vs Obj)',
    'loss_rpn_box_reg': 'RPN Regressão de Âncoras'
}

for k, color in colors.items():
    ax2.plot(range(1, NUM_EPOCHS + 1), loss_components[k], 'o--', label=labels_pt[k], color=color, linewidth=2)

ax2.set_title('Decomposição das 4 Perdas do Faster R-CNN', fontsize=12, fontweight='bold', color='#0A345D')
ax2.set_xlabel('Época', fontsize=11)
ax2.set_ylabel('Perda Componente', fontsize=11)
ax2.set_xticks(range(1, NUM_EPOCHS + 1))
ax2.legend(frameon=True, facecolor='white', framealpha=0.9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 10. Motor de Inferência Visual e Renderização de Bounding Boxes

Implementamos a função `run_inference_and_visualize()` que:
1. Executa o modelo em modo `eval()`.
2. Filtra as predições com base no limiar de confiança (`score_threshold`, ex: $\ge 0.70$).
3. Renderiza caixas delimitadoras coloridas de acordo com a fruta prevista (Vermelho = Maçã, Amarelo = Banana, Laranja = Laranja).
4. Plota o resultado previsto lado a lado com a anotação real (*Ground Truth*).


In [ ]:
def run_inference_and_visualize(
    model, 
    image_tensor, 
    ground_truth=None, 
    score_threshold=0.70, 
    class_names=CLASSES_PT,
    colors_map=CLASS_COLORS,
    title="Resultado da Inferência Faster R-CNN"
):
    """
    Executa a inferência no modelo em modo eval, filtra predições por limiar de confiança
    e renderiza o resultado lado a lado com o Ground Truth.
    """
    model.eval()
    with torch.no_grad():
        img_input = image_tensor.unsqueeze(0).to(device)
        predictions = model(img_input)[0]

    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    
    # Filtrar predições por Score Threshold
    boxes = predictions['boxes'].cpu().numpy()
    scores = predictions['scores'].cpu().numpy()
    labels = predictions['labels'].cpu().numpy()
    
    keep_indices = np.where(scores >= score_threshold)[0]
    filtered_boxes = boxes[keep_indices]
    filtered_scores = scores[keep_indices]
    filtered_labels = labels[keep_indices]
    
    has_gt = ground_truth is not None
    ncols = 2 if has_gt else 1
    fig, axes = plt.subplots(1, ncols, figsize=(8 * ncols, 6))
    if not has_gt:
        axes = [axes]
        
    # --- PLOT 1: Predições do Modelo ---
    ax_pred = axes[0]
    ax_pred.imshow(img_np)
    
    for box, score, label in zip(filtered_boxes, filtered_scores, filtered_labels):
        xmin, ymin, xmax, ymax = box
        width = xmax - xmin
        height = ymax - ymin
        
        color = colors_map.get(int(label), '#00D4FF')
        class_str = class_names.get(int(label), f"Classe {label}")
        text_label = f"{class_str} • {score * 100:.1f}%"
        
        # Bounding box prevista
        rect = patches.Rectangle(
            (xmin, ymin), width, height,
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax_pred.add_patch(rect)
        
        # Badge com nome da fruta e confiança
        ax_pred.text(
            xmin, ymin - 7,
            text_label,
            fontsize=9.5, fontweight='bold', color='#FFFFFF',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='#0A345D', edgecolor=color, linewidth=1.5, alpha=0.95)
        )
        
    ax_pred.set_title(f"🎯 Predições do Modelo (Score ≥ {score_threshold*100:.0f}%) • {len(filtered_boxes)} objeto(s)", fontsize=11, fontweight='bold', color='#0A345D')
    ax_pred.axis('off')
    
    # --- PLOT 2: Ground Truth Real (Comparação) ---
    if has_gt:
        ax_gt = axes[1]
        ax_gt.imshow(img_np)
        gt_boxes = ground_truth['boxes'].cpu().numpy()
        gt_labels = ground_truth['labels'].cpu().numpy()
        
        for box, label in zip(gt_boxes, gt_labels):
            xmin, ymin, xmax, ymax = box
            width = xmax - xmin
            height = ymax - ymin
            color = colors_map.get(int(label), '#22C55E')
            class_str = class_names.get(int(label), f"Classe {label}")
            
            rect = patches.Rectangle(
                (xmin, ymin), width, height,
                linewidth=2.5, edgecolor=color, facecolor='none'
            )
            ax_gt.add_patch(rect)
            ax_gt.text(
                xmin, ymin - 7,
                f"{class_str} (Real)",
                fontsize=9.5, fontweight='bold', color='#FFFFFF',
                bbox=dict(boxstyle='round,pad=0.25', facecolor=color, edgecolor='none', alpha=0.95)
            )
        ax_gt.set_title(f"📌 Ground Truth (Anotação Real) • {len(gt_boxes)} objeto(s)", fontsize=11, fontweight='bold', color='#15803D')
        ax_gt.axis('off')
        
    plt.suptitle(title, fontsize=13, fontweight='bold', color='#0A345D', y=0.98)
    plt.tight_layout()
    plt.show()
    
    return filtered_boxes, filtered_scores, filtered_labels

print("✅ Motor de inferência e renderização visual configurado!")


## 11. Avaliação Visual em Amostras do Conjunto de Teste

Vamos testar o modelo treinado em diferentes imagens do conjunto de teste contendo maçãs, bananas e laranjas isoladas e misturadas.


In [ ]:
# Avaliar amostras variadas do conjunto de teste
for sample_idx in [2, 15, 30, 45]:
    test_img, test_gt = test_dataset[sample_idx]
    run_inference_and_visualize(
        model=model,
        image_tensor=test_img,
        ground_truth=test_gt,
        score_threshold=0.70,
        title=f"Avaliação no Conjunto de Teste — Amostra #{sample_idx}"
    )


## 12. Como o Algoritmo NMS (Não-Supressão Máxima) Funciona por Baixo dos Panos

Quando a RPN gera centenas de âncoras candidatas, múltiplos retângulos sobrepostos acabam cobrindo a mesma fruta.

O algoritmo **NMS (`torchvision.ops.nms`)** resolve a redundância:
1. Ordena todas as caixas pela pontuação de confiança decrescente.
2. Seleciona a caixa de maior pontuação e a preserva na lista final.
3. Calcula o **IoU (Intersection over Union)** com todas as outras caixas da mesma classe:
   $$\text{IoU} = \frac{\text{Área da Interseção}}{\text{Área da União}}$$
   - Se $\text{IoU} > \text{IoU\_Threshold}$ (ex: $0.5$), a caixa sobreposta é **descartada** (*suprimida*).
4. Repete o processo até que todas as caixas tenham sido processadas.

Vamos demonstrar o operador `torchvision.ops.nms` em ação:


In [ ]:
from torchvision.ops import nms

# Simulação: 3 caixas altamente sobrepostas cobrindo a mesma maçã + 1 caixa cobrindo uma banana
sim_boxes = torch.tensor([
    [30.0, 40.0, 180.0, 200.0],  # Caixa 1 (Maçã): Alta confiança (0.96)
    [28.0, 42.0, 182.0, 198.0],  # Caixa 2 (Maçã): Quase idêntica à Caixa 1 (0.89)
    [32.0, 38.0, 178.0, 204.0],  # Caixa 3 (Maçã): Quase idêntica à Caixa 1 (0.75)
    [250.0, 60.0, 380.0, 310.0]  # Caixa 4 (Banana): Outra fruta separada (0.92)
])
sim_scores = torch.tensor([0.96, 0.89, 0.75, 0.92])

# Executar NMS com IoU Threshold de 0.50
keep_indices = nms(sim_boxes, sim_scores, iou_threshold=0.50)

print(f"📦 Total de Caixas Iniciais com sobreposição: {len(sim_boxes)}")
print(f"✅ Caixas Mantidas pós-NMS: {len(keep_indices)} (Índices: {keep_indices.tolist()})")
print(f"   • Caixa #{keep_indices[0].item()}: Score {sim_scores[keep_indices[0]].item():.2f} (Maçã)")
print(f"   • Caixa #{keep_indices[1].item()}: Score {sim_scores[keep_indices[1]].item():.2f} (Banana)")


## 13. 🎓 Desafios Práticos & Exercícios Propostos (Pós-Graduação)

Para aprofundar suas competências práticas em detecção de objetos:

---

### 🏋️‍♂️ Desafio 1: Análise de Trade-off (Score Threshold)
Varie o `score_threshold` entre `[0.20, 0.40, 0.60, 0.80, 0.95]` nas imagens de teste. Observe o comportamento de **Precisão vs Revocação** (limiares baixos capturam objetos difíceis mas introduzem falso-positivos; limiares altos geram detecções limpas mas podem omitir frutas menores).

### 🏋️‍♂️ Desafio 2: Baixar outro Dataset do Kaggle via `kagglehub`
Explore outros datasets de detecção no Kaggle utilizando `kagglehub.dataset_download()`, como:
- `andrewmvd/face-mask-detection` (Detecção de Máscaras com 3 classes: com máscara, sem máscara, máscara incorreta).
- `smeschke/four-shapes` (Detecção de formas geométricas).
Adapte a função `get_faster_rcnn_multiclass()` e treine o modelo no novo domínio!

### 🏋️‍♂️ Desafio 3: Comparação com One-Stage (Faster R-CNN vs YOLO)
Compare as métricas e a velocidade de inferência (FPS) do Faster R-CNN com um modelo One-Stage da família YOLO. Em quais cenários industriais a precisão cirúrgica do Faster R-CNN supera a velocidade do YOLO?
